https://docs.pyrogram.org/api/methods/

In [1]:
!pip install -q pyrogram tgcrypto pandas

In [37]:
import yaml
import logging
from pyrogram import Client
import pandas as pd
from google.colab import files

Мы сделали отдельный файл с конфигурациями в качестве "файла настроек", config.yaml, туда вынесены параметры по логированию и по телеграмму. Ну, по крайней мере, мы так поняли задание.. Сначала думали определить через словарь, но для корректного выполнения задания вынесли в отдельный файл. Для скрапинга тоже есть конфиг.

> Для получения высоких баллов добавьте в ваш проект логирование. Потренируйтесь включать и выключать его через файл настроек, а также задавать уровень логирования через этот же файл (не в приложении)



In [4]:
config_yaml = """
logging:
  level: INFO
  log_file: tg_collector.log

telegram:
  api_id: ****
  api_hash: ***
  phone: "***"
  channels:
    - data_hr
    - devops_jobs_feed
    - python_jobs
    - zrabota
    - vacancy_cs
    - analysts_hunter
"""

with open("config.yaml", "w") as f:
  f.write(config_yaml)

In [5]:
with open("config.yaml", "r") as f:
  config = yaml.safe_load(f)

logging https://docs.python.org/3/library/logging.html

In [6]:
def setup_logger(config: dict) -> logging.Logger:
  logging.root.handlers = []
  level = getattr(logging, config["logging"]["level"])
  logging.basicConfig(
      level=level,
      format="%(asctime)s [%(levelname)s] %(message)s",
      handlers=[
          logging.FileHandler(config["logging"]["log_file"], encoding="utf-8"),
          logging.StreamHandler()
      ]
  )
  return logging.getLogger("tg_collector")

In [7]:
logger = setup_logger(config)
logger.info("Логгер настроен")

2026-04-10 13:59:54,707 [INFO] Логгер настроен


pyrogram https://docs.pyrogram.org/api/client

In [8]:
app = Client(
    "vacancy_session",
    api_id=config["telegram"]["api_id"],
    api_hash=config["telegram"]["api_hash"],
    phone_number=config["telegram"]["phone"]
    )

In [9]:
await app.start()
logger.info("Pyrogram клиент запущен")

2026-04-10 13:59:59,909 [INFO] Connecting...
2026-04-10 14:00:00,077 [INFO] Connected! Production DC2 - IPv4
2026-04-10 14:00:00,080 [INFO] NetworkTask started
2026-04-10 14:00:01,021 [INFO] Session initialized: Layer 158
2026-04-10 14:00:01,025 [INFO] Device: CPython 3.12.13 - Pyrogram 2.0.106
2026-04-10 14:00:01,028 [INFO] System: Linux 6.6.113+ (en)
2026-04-10 14:00:01,028 [INFO] Session started
2026-04-10 14:00:01,168 [INFO] PingTask started
2026-04-10 14:00:01,571 [INFO] Started 6 HandlerTasks
2026-04-10 14:00:01,573 [INFO] Pyrogram клиент запущен


In [10]:
channels_info = []

Ниже была проверка, чтобы убедиться, что все работает и заодно посмотреть, что из себя представляет класс Chat наглядно – тут если что информация по нему https://docs.pyrogram.org/api/types/Chat

In [14]:
chat = await app.get_chat("python_jobs")
dir(chat)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_client',
 '_parse',
 '_parse_channel_chat',
 '_parse_chat',
 '_parse_chat_chat',
 '_parse_dialog',
 '_parse_full',
 '_parse_user_chat',
 'add_members',
 'archive',
 'available_reactions',
 'ban_member',
 'bind',
 'bio',
 'can_set_sticker_set',
 'dc_id',
 'default',
 'description',
 'distance',
 'export_invite_link',
 'first_name',
 'get_member',
 'get_members',
 'has_protected_content',
 'id',
 'invite_link',
 'is_creator',
 'is_fake',
 'is_restricted',
 'is_scam',
 'is_support',
 'is_verified',
 'join',
 'last_name',
 'leave',
 'linked_chat',
 'mark_unread',
 'members_count',
 'permiss

In [15]:
chat

pyrogram.types.Chat(id=-1001067529236, type=pyrogram.enums.ChatType.SUPERGROUP, is_verified=False, is_restricted=False, is_creator=False, is_scam=False, is_fake=False, title='Python — вакансии и аналитика', username='python_jobs', photo=pyrogram.types.ChatPhoto(small_file_id='AQADAgADuKcxG3FWIRAAEAIAA-y7uesW____GEFQYGvw7qcABB4E', small_photo_unique_id='AgADuKcxG3FWIRA', big_file_id='AQADAgADuKcxG3FWIRAAEAMAA-y7uesW____GEFQYGvw7qcABB4E', big_photo_unique_id='AgADuKcxG3FWIRA'), description='Публикуем вакансии и запросы на поиск работы по направлению Python, Flask и т.д.\n\nЗдесь всё: full-time, part-time, remote и разовые подработки.\n\nСм. также: @golang_jobs, @qa_jobs, @devops_jobs, @javascript_jobs, @nodejs_jobs, @uiux_jobs, @products_jobs', dc_id=2, has_protected_content=False, pinned_message=pyrogram.types.Message(id=238422, from_user=pyrogram.types.User(id=614127860, is_self=False, is_contact=False, is_mutual_contact=False, is_deleted=False, is_bot=False, is_verified=False, is_rest

# 1 запрос - get_chat()

Мы используем его, чтобы получить метаданные по каждому каналу - канал (тг айди), название, описание и количество участников. Это даст нам представление о канале.

In [16]:
for channel in config["telegram"]["channels"]:
  try:
    chat = await app.get_chat(channel)
    info = {
        "channel": channel,
        "title": chat.title,
        "description": chat.description,
        "members_count": chat.members_count,
    }
    channels_info.append(info)
    logger.info(f"get_chat({channel}): {chat.title}, {chat.members_count} подписчиков")
  except Exception:
    logger.warning(f"Ошибка {Exception}! get_chat() для {channel} не получен")

2026-04-10 14:00:49,767 [INFO] get_chat(data_hr): Data Analytics Jobs | Вакансии, 9921 подписчиков
2026-04-10 14:00:50,208 [INFO] get_chat(devops_jobs_feed): Devops Jobs — вакансии и резюме, 19828 подписчиков
2026-04-10 14:00:50,717 [INFO] get_chat(python_jobs): Python — вакансии и аналитика, 7454 подписчиков
2026-04-10 14:00:51,280 [INFO] get_chat(zrabota): АйТи Вакансии. ИИ Работа. Фриланс. Удаленка, 21733 подписчиков
2026-04-10 14:00:52,262 [INFO] get_chat(vacancy_cs): ФКН: Вакансии, 14481 подписчиков
2026-04-10 14:00:53,419 [INFO] get_chat(analysts_hunter): Работа ищет аналитиков // Вакансии, 25060 подписчиков


In [17]:
df_channels = pd.DataFrame(channels_info)
df_channels.to_csv("channels_info.csv", index=False, encoding="utf-8-sig")
logger.info(f"Информация о {len(df_channels)} каналах сохранена")
df_channels

2026-04-10 14:00:55,624 [INFO] Информация о 6 каналах сохранена


,channel,title,description,members_count
0,data_hr,Data Analytics Jobs | Вакансии,Канал с вакансиями для специалистов по работе ...,9921
1,devops_jobs_feed,Devops Jobs — вакансии и резюме,Вакансии из @devops_jobs\n\nОстальные каналы: ...,19828
2,python_jobs,Python — вакансии и аналитика,Публикуем вакансии и запросы на поиск работы п...,7454
3,zrabota,АйТи Вакансии. ИИ Работа. Фриланс. Удаленка,Свежие АйТи/Диджитал вакансии каждый день по р...,21733
4,vacancy_cs,ФКН: Вакансии,"Канал с вакансиями, мероприятиями и хакатонами...",14481
5,analysts_hunter,Работа ищет аналитиков // Вакансии,Правила размещения вакансий: https://teletype....,25060


In [18]:
logger.info(f"Запрос 1 'get_chat()' успешно выполнен для {len(df_channels)} каналов")

2026-04-10 14:00:57,728 [INFO] Запрос 1 'get_chat()' успешно выполнен для 6 каналов


# Запрос 2 - get_chat_history()
Данный запрос выдает нам историю чата, то есть это наш основной запрос, по которому мы собираем сообщения о вакансиях. Также уточню, что у нас есть лимит по сообранным сообщениям на канал, мы установили 5000 - это нужно для того, чтобы ???. Мы собираем сообщения в таком формате: канал, айди сообщения, дата публикации, сам текст и указываем источник сбора "телеграм".

In [19]:
all_vacancies = []

In [20]:
for channel in config["telegram"]["channels"]:
  logger.info(f"Канал {channel}:")
  channel_count = 0
  try:
    async for message in app.get_chat_history(channel, limit=5000):
      text = message.text or message.caption
      if not text:
        continue
      vacancy = {
          "channel": channel,
          "message_id": message.id,
          "date": message.date,
          "text": text,
          "source": "telegram",
          }
      all_vacancies.append(vacancy)
      channel_count += 1
    logger.info(f"В канале {channel} собрано {channel_count} сообщений")
  except Exception:
    logger.error(f"Ошибка {Exception}! get_chat_history() для {channel} не получен")

2026-04-10 14:01:47,777 [INFO] Канал data_hr:
2026-04-10 14:01:54,155 [INFO] В канале data_hr собрано 1147 сообщений
2026-04-10 14:01:54,156 [INFO] Канал devops_jobs_feed:
2026-04-10 14:02:07,828 [WARNING] [vacancy_session] Waiting for 18 seconds before continuing (required by "messages.GetHistory")
2026-04-10 14:02:35,705 [INFO] В канале devops_jobs_feed собрано 5000 сообщений
2026-04-10 14:02:35,706 [INFO] Канал python_jobs:
2026-04-10 14:02:51,779 [WARNING] [vacancy_session] Waiting for 15 seconds before continuing (required by "messages.GetHistory")
2026-04-10 14:03:17,215 [INFO] В канале python_jobs собрано 4687 сообщений
2026-04-10 14:03:17,216 [INFO] Канал zrabota:
2026-04-10 14:03:25,106 [INFO] В канале zrabota собрано 1177 сообщений
2026-04-10 14:03:25,107 [INFO] Канал vacancy_cs:
2026-04-10 14:03:30,454 [INFO] В канале vacancy_cs собрано 729 сообщений
2026-04-10 14:03:30,455 [INFO] Канал analysts_hunter:
2026-04-10 14:03:36,837 [WARNING] [vacancy_session] Waiting for 3 second

In [21]:
df_vacancies = pd.DataFrame(all_vacancies)
df_vacancies.to_csv("vacancies_tg_main.csv", index=False, encoding="utf-8-sig")
logger.info(f"Успешно собрано {len(df_vacancies)} сообщений")
df_vacancies

2026-04-10 14:04:41,646 [INFO] Успешно собрано 17452 сообщений


,channel,message_id,date,text,source
0,data_hr,1290,2026-04-09 09:36:02,LADA Цифра — дочерняя IT-компания АО «АВТОВАЗ»...,telegram
1,data_hr,1289,2026-04-07 11:22:32,🚀 Учись работать с реальными продуктовыми зада...,telegram
2,data_hr,1287,2026-04-03 13:25:54,#вакансия \n\nLead/Senior Data Scientist\n\nНа...,telegram
3,data_hr,1285,2026-03-24 08:57:38,#вакансия #разработчикDWH \n\nКомпания: Зонтик...,telegram
4,data_hr,1284,2026-03-20 09:49:22,#вакансия #аналитикданных\n🔎 Аналитик данных\n...,telegram
...,...,...,...,...,...
17447,analysts_hunter,410118,2026-02-05 11:25:23,А лиду задавать вопросы можно?,telegram
17448,analysts_hunter,410117,2026-02-05 11:25:07,Заводить второй акк и писать стейкхолдерам с н...,telegram
17449,analysts_hunter,410116,2026-02-05 11:24:59,"Нет у нас ответа. Возможно и вопросы глупые ,и...",telegram
17450,analysts_hunter,410115,2026-02-05 11:23:11,"Вопрос четверга от друга. Что делать, если лид...",telegram


In [22]:
logger.info(f"Запрос 2 'get_chat_history()' успешно выполнен, итого {len(df_vacancies)} сообщений")

2026-04-10 14:04:50,325 [INFO] Запрос 2 'get_chat_history()' успешно выполнен, итого 17452 сообщений


https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.contains.html

In [24]:
mask = df_vacancies["text"].str.contains("#вакансия|#vacancy|#job|работа|вакансия|зарплата|зп|оплата|₽|руб", case=False, na=False)
print(f"Всего сообщений {len(df_vacancies)}, из них скорее всего вакансии {mask.sum()}, скорее всего мусор {(~mask).sum()}")

Всего сообщений 17452, из них скорее всего вакансии 8460, скорее всего мусор 8992


# 3 запрос - search_messages()

In [25]:
search_terms = [
    "Python",
    "Java",
    "Data Scientist",
    "DS",
    "Analyst",
    "Data Engineer",
    "Machine Learning",
    "ML",
    "DL",
    "DevOps",
    "Junior",
    "Middle",
    "Senior"
    ]
search_results = []

In [26]:
for channel in config["telegram"]["channels"]:
  for term in search_terms:
    try:
      async for message in app.search_messages(
          channel,
          query=term,
          limit=150
      ):
        text = message.text or message.caption
        if not text:
          continue
        vacancy = {
            "channel": channel,
            "message_id": message.id,
            "date": message.date,
            "text": text,
            "search_term": term,
            "source": "telegram",
            }
        search_results.append(vacancy)
      logger.info(f"Успешно найдены сообщения с {term} в канале {channel}")
    except Exception:
      logger.warning(f"Ошибка {Exception}! search_messages() для {channel} с {term} не получен")

2026-04-10 14:10:35,040 [INFO] Успешно найдены сообщения с Python в канале data_hr
2026-04-10 14:10:35,476 [INFO] Успешно найдены сообщения с Java в канале data_hr
2026-04-10 14:10:36,174 [INFO] Успешно найдены сообщения с Data Scientist в канале data_hr
2026-04-10 14:10:37,149 [INFO] Успешно найдены сообщения с DS в канале data_hr
2026-04-10 14:10:37,924 [INFO] Успешно найдены сообщения с Analyst в канале data_hr
2026-04-10 14:10:38,667 [INFO] Успешно найдены сообщения с Data Engineer в канале data_hr
2026-04-10 14:10:39,160 [INFO] Успешно найдены сообщения с Machine Learning в канале data_hr
2026-04-10 14:10:40,103 [INFO] Успешно найдены сообщения с ML в канале data_hr
2026-04-10 14:10:40,619 [INFO] Успешно найдены сообщения с DL в канале data_hr
2026-04-10 14:10:41,212 [INFO] Успешно найдены сообщения с DevOps в канале data_hr
2026-04-10 14:10:41,884 [INFO] Успешно найдены сообщения с Junior в канале data_hr
2026-04-10 14:10:42,705 [INFO] Успешно найдены сообщения с Middle в канале 

In [27]:
df_search = pd.DataFrame(search_results)
df_search.to_csv("vacancies_tg_terms.csv", index=False, encoding="utf-8-sig")
logger.info(f"Успешно собрано {len(df_search)} сообщений")
df_search

2026-04-10 14:20:22,480 [INFO] Успешно собрано 7396 сообщений


,channel,message_id,date,text,search_term,source
0,data_hr,1290,2026-04-09 09:36:02,LADA Цифра — дочерняя IT-компания АО «АВТОВАЗ»...,Python,telegram
1,data_hr,1289,2026-04-07 11:22:32,🚀 Учись работать с реальными продуктовыми зада...,Python,telegram
2,data_hr,1287,2026-04-03 13:25:54,#вакансия \n\nLead/Senior Data Scientist\n\nНа...,Python,telegram
3,data_hr,1284,2026-03-20 09:49:22,#вакансия #аналитикданных\n🔎 Аналитик данных\n...,Python,telegram
4,data_hr,1283,2026-03-20 07:34:07,Всем привет!\n\nКомпания: Большой зелёный банк...,Python,telegram
...,...,...,...,...,...,...
7391,analysts_hunter,394837,2025-08-08 08:26:39,\#вакансия #systemanalyst #работа #hiring #job...,Senior,telegram
7392,analysts_hunter,394767,2025-08-07 12:21:24,#vacancy #вакансия\n\n💙 Компания: YADRO \n🔗 Ва...,Senior,telegram
7393,analysts_hunter,394677,2025-08-06 14:16:31,#вакансия #аналитика #работа #hiring #job #sen...,Senior,telegram
7394,analysts_hunter,394282,2025-08-05 10:59:59,#вакансия #SQL #работа #hiring #job #middle #s...,Senior,telegram


In [28]:
logger.info(f"Запрос 3 'search_messages()' успешно выполнен, итого {len(df_search)} записей")

2026-04-10 14:20:40,200 [INFO] Запрос 3 'search_messages()' успешно выполнен, итого 7396 записей


# Запрос 4

In [29]:
history_counts = []

In [30]:
for channel in config["telegram"]["channels"]:
  try:
    count = await app.get_chat_history_count(channel)
    history_count = {
        "channel": channel,
        "total_messages": count,
        }
    history_counts.append(history_count)
    logger.info(f"В канале {channel} всего {count} сообщений")
  except Exception:
    logger.error(f"Ошибка {Exception}! get_chat_history_count() для {channel} не получен")

2026-04-10 14:21:26,786 [INFO] В канале data_hr всего 1172 сообщений
2026-04-10 14:21:26,974 [INFO] В канале devops_jobs_feed всего 22249 сообщений
2026-04-10 14:21:27,173 [INFO] В канале python_jobs всего 187360 сообщений
2026-04-10 14:21:27,358 [INFO] В канале zrabota всего 1193 сообщений
2026-04-10 14:21:27,760 [INFO] В канале vacancy_cs всего 748 сообщений
2026-04-10 14:21:27,944 [INFO] В канале analysts_hunter всего 191496 сообщений


In [31]:
df_counts = pd.DataFrame(history_counts)
df_counts.to_csv("total_messages_count.csv", index=False, encoding="utf-8-sig")
logger.info(f"Успешно собрано количество сообщений для {len(df_counts)} каналов")
df_counts

2026-04-10 14:22:05,222 [INFO] Успешно собрано количество сообщений для 6 каналов


,channel,total_messages
0,data_hr,1172
1,devops_jobs_feed,22249
2,python_jobs,187360
3,zrabota,1193
4,vacancy_cs,748
5,analysts_hunter,191496


# Запрос 5

In [32]:
high_salary_vacancies = []

In [33]:
mask = df_vacancies["text"].str.contains(r"[4-9]\d{2}[\s\.]?000|[4-9]\d{2}\s?[кkКK]", case=False, na=False)
df_high = df_vacancies[mask]
logger.info(f"Найдено {len(df_high)} вакансий с зарплатой от 400.000 руб")

2026-04-10 14:22:48,694 [INFO] Найдено 911 вакансий с зарплатой от 400.000 руб


In [34]:
for channel in config["telegram"]["channels"]:
  channel_ids = (df_high[df_high["channel"] == channel]["message_id"].tolist())
  if not channel_ids:
    continue
  batch_size = 200
  batches = [channel_ids[i:i+batch_size] for i in range(0, len(channel_ids), batch_size)]
  for batch in batches:
    try:
      messages = await app.get_messages(channel, batch)
      for message in messages:
        text = message.text or message.caption
        if not text:
          continue
        vacancy = {
            "channel": channel,
            "message_id": message.id,
            "date": message.date,
            "text": text,
            "source": "telegram",
            }
        high_salary_vacancies.append(vacancy)
      logger.info(f"Для канала {channel} успешно найден get_messages() для вакансий с зарплатой 400к+")
    except Exception as e:
          logger.error(f"Ошибка {e}! get_messages() для {channel} не получен")

2026-04-10 14:28:01,700 [INFO] Для канала data_hr успешно найден get_messages() для вакансий с зарплатой 400к+
2026-04-10 14:28:02,292 [INFO] Для канала devops_jobs_feed успешно найден get_messages() для вакансий с зарплатой 400к+
2026-04-10 14:28:02,564 [WARNING] [vacancy_session] Waiting for 30 seconds before continuing (required by "channels.GetMessages")
2026-04-10 14:28:33,146 [INFO] Для канала devops_jobs_feed успешно найден get_messages() для вакансий с зарплатой 400к+
2026-04-10 14:28:33,324 [WARNING] [vacancy_session] Waiting for 30 seconds before continuing (required by "channels.GetMessages")
2026-04-10 14:29:03,978 [INFO] Для канала devops_jobs_feed успешно найден get_messages() для вакансий с зарплатой 400к+
2026-04-10 14:29:04,156 [WARNING] [vacancy_session] Waiting for 30 seconds before continuing (required by "channels.GetMessages")
2026-04-10 14:29:34,674 [INFO] Для канала devops_jobs_feed успешно найден get_messages() для вакансий с зарплатой 400к+
2026-04-10 14:29:35

In [35]:
df_high_salary = pd.DataFrame(high_salary_vacancies)
df_high_salary.to_csv("vacancies_tg_high_salary.csv", index=False, encoding="utf-8-sig")
logger.info(f"Успешно собрано {len(df_high_salary)} вакансий с зарплатой 400к+")
df_high_salary

2026-04-10 14:33:01,331 [INFO] Успешно собрано 911 вакансий с зарплатой 400к+


,channel,message_id,date,text,source
0,data_hr,1250,2026-01-26 08:25:47,Data Never Lies is a UK-based data consultancy...,telegram
1,data_hr,1242,2026-01-13 12:33:26,Data Never Lies is a UK-based data consultancy...,telegram
2,data_hr,1236,2025-12-25 14:16:39,#вакансия\nПозиция: Дата-аналитик (Graph Analy...,telegram
3,data_hr,1164,2025-10-08 14:49:13,#Вакансия #аналитик #Product #Productanalyst #...,telegram
4,data_hr,1135,2025-09-04 08:37:09,#работа #удаленная_работа #вакансия #Team_Lead...,telegram
...,...,...,...,...,...
906,analysts_hunter,412921,2026-03-10 12:41:39,"День добрый. Ищу работу веб - аналитиком, CRM ...",telegram
907,analysts_hunter,412742,2026-03-06 10:51:47,"Да ну достаточно на hh поиск запустить, чтобы ...",telegram
908,analysts_hunter,412566,2026-03-05 14:28:42,#вакансия #product #owner #senior\nПозиция: Pr...,telegram
909,analysts_hunter,411859,2026-02-25 08:36:08,Правила размещения вакансий в чате @analysts_h...,telegram


In [36]:
for f in [
    "config.yaml",
    "tg_collector.log",
    "channels_info.csv",
    "vacancies_tg_main.csv",
    "vacancies_tg_terms.csv",
    "vacancies_tg_high_salary.csv",
    "total_messages_count.csv",
    ]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>